# שיעור 13 - זיכרון סוכן


## התקנה

מחברת זו מדגימה כיצד לבנות סוכן הזמנת טיולים עם **זיכרון מתמיד** באמצעות **מסגרת הסוכן של מיקרוסופט** (MAF).

תלמד כיצד סוגי הזיכרון השונים של הסוכן — זיכרון עבודה, קצר טווח וארוך טווח — משפיעים על האופן שבו הסוכן שומר ומשתמש במידע לאורך שיחות.

**דרישות מוקדמות:**
- פרויקט Microsoft Foundry עם מודל צ'אט מופעל (למשל `gpt-4.1-mini`).
- מחובר באמצעות Azure CLI — הרץ `az login` בטרמינל שלך.
- `AZURE_AI_PROJECT_ENDPOINT` — נקודת הקצה של פרויקט Microsoft Foundry שלך.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — שם המודל המופעל שלך.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## סוגי זיכרון של סוכן

סוכני AI יכולים להשתמש בסוגים שונים של זיכרון, כאשר כל אחד משרת מטרה שונה:

### זיכרון עבודה
שרשרת השיחה עצמה — ההודעות שהוחלפו במפגש יחיד. הסוכן יכול להתייחס להודעות קודמות באותה שרשרת כדי לשמור על קוהרנטיות. ב-MAF זה נוצר דרך **`agent.create_session()`**, שמחזיר `AgentSession`.

### זיכרון קצר טווח
מידע שנשמר לאורך משימה או מפגש אך אינו מאוחסן לצמיתות. לדוגמה, הסוכן עשוי לאסוף עובדות במהלך שיחה תכנונית מרובת סבבים ולהשתמש בהן ליצירת מסלול סופי.

### זיכרון ארוך טווח
העדפות ועובדות שנשמרות **בין מפגשים**. משתמש חוזר לא אמור לחזור על מגבלות התזונה שלו או סגנון הנסיעה שלו. זיכרון ארוך טווח בדרך כלל מגובה על ידי מאגר חיצוני — מסד נתונים, קובץ, או אינדקס וקטורי — ומוצג לסוכן דרך כלים.


## זיכרון עבודה עם סשנים

הצורה הפשוטה ביותר של זיכרון היא סשן השיחה. כאשר מעבירים את אותו הסשן (נוצר באמצעות `agent.create_session()`) לקריאות בשרשור של `agent.run()`, הסוכן רואה את ההיסטוריה המלאה של אותה שיחה ויכול לזכור פרטים מוקדמים.

בואו ניצור סוכן נסיעות ונדגים זיכרון עבודה.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

הסוכן זכר נכון את התקציב כי שתי ההודעות שייכות לאותה סשן. זו **זיכרון עבודה** — הוא קיים רק למשך חיי הסשן.

### מה קורה עם אשכול חדש?

אם ניצור סשן **חדש**, לסוכן אין זיכרון של השיחה הקודמת:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## תבנית זיכרון לטווח ארוך

כדי לזכור העדפות משתמש **בין הפעלות**, אנחנו צריכים מקום אחסון מתמיד שיחיה מחוץ לשרשור השיחה. הסוכן ניגש למקום אחסון זה דרך **כלים** — פונקציות שהוא יכול לקרוא כדי לשמור ולהחזיר מידע.

מטה מיישמים מחסן העדפות פשוט בזיכרון (ביישום פרודקשן היית מגבה את זה עם בסיס נתונים או אינדקס וקטורי) ומציגים אותו ככלים שהסוכן יכול להשתמש בהם.

### ארכיטקטורה
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### תרחיש 1 — משתמש בפעם הראשונה מזמין טיול יום השנה

שרה מבקרת בפעם הראשונה. הסוכן צריך לשמור את ההעדפות שלה דרך הכלים ולהשתמש בהן כדי להמליץ על בתי מלון.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### תרחיש 2 — שרה חוזרת לאחר שבועות

שרה מתחילה **שרשור חדש לגמרי** (מדמה סשן חדש). זיכרון העבודה ריק, אך מאגר ההעדפות לטווח הארוך עדיין מכיל את המידע שלה. הסוכן צריך לאחזר את המידע ולהשתמש בו כדי להתאים אישית את ההמלצות.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## סיכום

בשיעור זה למדת שלושה סוגי זיכרון של סוכן ואיך ליישם אותם עם Microsoft Agent Framework:

| סוג זיכרון | מנגנון MAF | משך חיים |
|---|---|---|
| **עובד** | `agent.create_session()` | שיחה אחת |
| **קצר טווח** | הקשר מצטבר בתוך אשף | משימה / מושב אחד |
| **ארוך טווח** | מאגר חיצוני שאליו ניגשים באמצעות פונקציות `@tool` | בין מושבים |

### תובנות מרכזיות
1. **`agent.create_session()`** מספק זיכרון עובד — הסוכן רואה את כל היסטוריית השיחה בתוך מושב.
2. **מושבים חדשים מאבדים הקשר** — בלי זיכרון ארוך טווח הסוכן לא יכול לזכור שיחות עבר.
3. **פונקציות `@tool`** גשר המחבר — הן מאפשרות לסוכן לשמור ולשלוף מידע מתוך מאגר מתמשך.
4. **התאמה אישית משתפרת עם הזמן** — ככל שנשמרים יותר העדפות, ההמלצות של הסוכן טובות יותר.

### יישומים בעולם האמיתי
- **שירות לקוחות**: זיכרון היסטוריית הלקוח והעדפותיו
- **עוזרים אישיים**: שמירת הקשר לאורך ימים או שבועות
- **בריאות**: מעקב אחר מידע והעדפות של המטופל
- **מסחר אלקטרוני**: קנייה מותאמת אישית על בסיס היסטוריה

### שלבים הבאים
- החלף את מילון הזיכרון בזיכרון מבוסס מסד נתונים או מאגר וקטורים (למשל Azure AI Search)
- הוסף תוקף לזיכרון עבור מידע רגיש לזמן
- בנה מערכות רב-סוכן עם זיכרון משותף
- חקור את מחברת Cognee לזיכרון מגובה גרף ידע


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
